# Выполненное автономное продолжение

286 контрактов → lexical/dense поиск → двунаправленный NLI → условная
проверка числовых отношений. Отдельно: чтение полных terms и межплощадочный кейс.

Этот notebook **читает и проверяет сохранённые результаты фактических запусков**.
Он не повторяет inference. Код запуска моделей находится в `extension/nlp/`.
Независимой человеческой разметки нет; никакие scores не являются market accuracy.


In [1]:
from pathlib import Path
import json, hashlib
from collections import Counter
root=Path.cwd()
if root.name=='notebooks': root=root.parent
nlp=root/'extension/nlp'
def read(path): return json.loads((root/path).read_text())
manifest=json.loads((nlp/'artifact_manifest.json').read_text())
for f in manifest['files']:
    p=nlp/f['path']
    assert hashlib.sha256(p.read_bytes()).hexdigest()==f['sha256'], str(p)
print('Проверено файлов:',len(manifest['files']))
corpus=read('extension/nlp/corpus_run_manifest.json')
assert 'completed_at_utc' in corpus
assert hashlib.sha256((root/'pilot/markets.json').read_bytes()).hexdigest()==corpus['input_hash']
print('NLI завершена:',corpus['completed_at_utc'])
print('Пар / направленных входов:',corpus['n_pairs'],corpus['directional_inferences'])


Проверено файлов: 23
NLI завершена: 2026-09-28T21:24:42.914177+00:00
Пар / направленных входов: 1188 2376


## 1. Что дал NLI по всему candidate pool

Каждая метка ниже — **предложение модели, verified=false**.
Не читать число exclusion как число правильно обнаруженных взаимоисключений.


In [2]:
for label,n in sorted(corpus['output_counts'].items()):
    print(f'{label:28s} {n:4d}')
assert sum(corpus['output_counts'].values())==1188
print('Усечённых входов:',corpus['truncated_inputs'])


a_implies_b_candidate          58
abstain                       485
b_implies_a_candidate          54
equivalence_candidate         148
exclusion_candidate           443
Усечённых входов: 0


## 2. Проверяемый числовой свидетель

Среди 839 пар одинаковых пороговых event-шаблонов есть 325 NLI-exclusion
кандидатов. Но в заданной общей неограниченной скалярной модели выбор значения
выше обоих порогов делает обе пропозиции истинными. Сравнение имеет силу внутри
этой модели; полные договорные выплаты оно не сертифицирует.


In [3]:
consistency=read('extension/nlp/conditional_consistency.json')
print('Совпало шаблонных пар:',consistency['matched_pairs'])
print('Групп:',consistency['event_template_groups'])
print('Конфликтов exclusion:',consistency['exclusion_conflicts'])
example=next(r for r in consistency['records'] if r['conflict_in_scalar_scope'])
print(json.dumps(example,ensure_ascii=False,indent=2))


Совпало шаблонных пар: 839
Групп: 19
Конфликтов exclusion: 325
{
  "a": "KXLCPIMAXYOY-27-P6.5",
  "b": "KXLCPIMAXYOY-27-P5",
  "nli_candidate_type": "exclusion_candidate",
  "template_group": "711357093fbdb7a7ff93a86b0045330cce1c404fb52ca20ce5bad794608f699c",
  "scalar_both_yes_witness": "7.5",
  "conflict_in_scalar_scope": true,
  "scope": "Same unrestricted scalar value, same template, binary rule-following settlement only.",
  "human_gold": false,
  "unconditional_contract_certified": false
}


## 3. Негативный результат улучшения retrieval

На двух post-hoc пороговых контролях k=10 lexical находит оба с 2012 парами.
k=5 + шаблоны тоже находит оба, но требует 3187 пар. Поэтому улучшение
шаблонного генератора при одинаковом бюджете **не установлено**.
Это не recall на неизвестных связях. [Полный протокол](../extension/retrieval/README.md).

## 4. Порядок eligibility-фильтра меняет бюджет


In [4]:
eligibility=read('extension/nlp/eligible_retrieval.json')
for kind in ['lexical','dense']:
    print(kind,eligibility['methods'][kind]['union']['5'])
print('Больше кандидатов не означает более высокую точность при равных затратах.')


lexical {'filter_before_topk_pairs': 1455, 'filter_after_topk_pairs': 349}
dense {'filter_before_topk_pairs': 1849, 'filter_after_topk_pairs': 19}
Больше кандидатов не означает более высокую точность при равных затратах.


## 5. Диагностика реальных пар

Выбранные заранее в первом пилоте примеры, три views. Ниже только title_rules
для двух кейсов; остальные результаты сохранены, а не скрыты.


In [5]:
pred=read('extension/nlp/nli_predictions.json')
for r in pred:
    if r['kind']=='real_exploratory' and r['view']=='title_rules' and r['id'] in ['different_periods','strict_thresholds']:
        print(r['id'],r['direction'],r['prediction'],
              {k:round(v,3) for k,v in r['probabilities'].items()})


strict_thresholds a_to_b contradiction {'entailment': 0.04, 'neutral': 0.012, 'contradiction': 0.948}
strict_thresholds b_to_a contradiction {'entailment': 0.029, 'neutral': 0.055, 'contradiction': 0.916}
different_periods a_to_b contradiction {'entailment': 0.024, 'neutral': 0.008, 'contradiction': 0.969}
different_periods b_to_a contradiction {'entailment': 0.084, 'neutral': 0.015, 'contradiction': 0.9}


## 6. Аудит собственной синтетической проверки

В v1 были дубли и неоднозначность real CPI / continuous toy domain. V2 сохраняет
историю, вводит вымышленные непрерывные показатели и дедуплицирует входы до
нового inference. Это поправка после анализа, не новый закрытый тест.


In [6]:
v2=read('extension/nlp/synthetic_v2_manifest.json')
print('Правильных / уникальных входов:',v2['correct'],v2['n_unique_ordered_text_pairs'])
print('Always-neutral на этом составе:',v2['gold_counts']['neutral'])
print('Матрица ошибок:')
for r in v2['confusion']:
    print(r['gold'],'->',r['prediction'],':',r['n'])
print('Шаблоны зависимы; это не точность на реальных рынках.')


Правильных / уникальных входов: 32 63
Always-neutral на этом составе: 33
Матрица ошибок:
contradiction -> contradiction : 18
entailment -> contradiction : 3
entailment -> entailment : 9
neutral -> contradiction : 23
neutral -> entailment : 5
neutral -> neutral : 5
Шаблоны зависимы; это не точность на реальных рынках.


## 7. Что изменилось после чтения документов

Полные Kalshi terms содержат разные fallback/expiry для месячного YoY и
любого месяца года. Поэтому обычное числовое следование требует scope.
Одна межплощадочная пара также найдена, но Polymarket rules взяты из поискового
индекса с непроверенной актуальностью; безусловный вывод отклонён.

- [Аудит полных условий](../extension/terms/AUDIT_RU.md)
- [Межплощадочный пример и качество источников](../extension/cross_platform/README.md)
- [Обновлённый исследовательский вывод](../EXPERIMENT_UPDATE_RU.md)

Главная гипотеза следующего этапа: отдельно измерить эффект полных документов,
структурирования и контрпримеров при равном бюджете и независимом human gold.
